# 02 — Baseline Recommenders: Popularity and Content-Based

Two simple models, used as baselines for the rest of the project:

1. **Popularity-based**: the same "top books" list for everyone, ranked with a weighted rating.
2. **Content-based**: for a given book, recommend the books whose authors and tags are the most similar,
   using TF-IDF vectors and cosine similarity.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

DATA = "../data"

books = pd.read_csv(f"{DATA}/books.csv")
tags = pd.read_csv(f"{DATA}/tags.csv")
book_tags = pd.read_csv(f"{DATA}/book_tags.csv")

## 1. Popularity-based recommender

Sorting by `average_rating` alone would favour obscure books with a handful of
5-star ratings. Instead, we use the **weighted rating** popularised by IMDb's
Top 250, which pulls the average of rarely-rated books towards the global mean:

$$
\text{score} = \frac{v}{v + m} \, R + \frac{m}{v + m} \, C
$$

- $R$: average rating of the book, $v$: its number of ratings
- $C$: mean of all books' average ratings
- $m$: minimum number of ratings to be trusted — here the 90th percentile of `ratings_count`

`average_rating` and `ratings_count` come from `books.csv` and reflect all Goodreads ratings,
not only the 6 million ratings of `ratings.csv`.

In [2]:
C = books["average_rating"].mean()
m = books["ratings_count"].quantile(0.90)
v, R = books["ratings_count"], books["average_rating"]

books["score"] = v / (v + m) * R + m / (v + m) * C

print(f"C = {C:.2f}, m = {m:,.0f} ratings")
books.sort_values("score", ascending=False)[["title", "authors", "average_rating", "ratings_count", "score"]].head(10)

C = 4.00, m = 94,103 ratings


,title,authors,average_rating,ratings_count,score
24,Harry Potter and the Deathly Hallows (Harry Po...,"J.K. Rowling, Mary GrandPré",4.61,1746574,4.578926
26,Harry Potter and the Half-Blood Prince (Harry ...,"J.K. Rowling, Mary GrandPré",4.54,1678823,4.511454
17,Harry Potter and the Prisoner of Azkaban (Harr...,"J.K. Rowling, Mary GrandPré, Rufus Beck",4.53,1832823,4.504224
23,Harry Potter and the Goblet of Fire (Harry Pot...,"J.K. Rowling, Mary GrandPré",4.53,1753043,4.503111
421,"Harry Potter Boxset (Harry Potter, #1-7)",J.K. Rowling,4.74,190050,4.495659
134,"A Storm of Swords (A Song of Ice and Fire, #3)",George R.R. Martin,4.54,469022,4.450127
191,The Name of the Wind (The Kingkiller Chronicle...,Patrick Rothfuss,4.55,400101,4.445690
20,Harry Potter and the Order of the Phoenix (Har...,"J.K. Rowling, Mary GrandPré",4.46,1735368,4.436452
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44,4602479,4.431228
160,"The Return of the King (The Lord of the Rings,...",J.R.R. Tolkien,4.51,463959,4.424371


The top 10 is dominated by a few series (*Harry Potter*, *A Song of Ice and
Fire*, *The Lord of the Rings*). It is a reasonable default for a new user, but
it is not personalised and keeps promoting the same bestsellers.

## 2. Content-based recommender

### 2.1 Building a text profile for each book

Each book is described by a "document" made of:
- its **authors**, each turned into a single token (`J.K._Rowling`) so that first and last names stay together;
- its **20 most frequent tags**, after removing tags that describe the reader's shelf rather than the book.

In [3]:
SHELF_TAGS = {"to-read", "currently-reading", "favorites", "owned",
              "books-i-own", "default", "library", "kindle"}

bt = book_tags.merge(tags, on="tag_id")
bt = bt[~bt["tag_name"].isin(SHELF_TAGS)]
bt = bt.sort_values("count", ascending=False).groupby("goodreads_book_id").head(20)
tag_text = bt.groupby("goodreads_book_id")["tag_name"].apply(" ".join)

books["tags"] = books["goodreads_book_id"].map(tag_text).fillna("")
author_tokens = books["authors"].str.replace(" ", "_").str.replace(",_", " ")
books["content"] = author_tokens + " " + books["tags"]

books.loc[0, "content"]

'Suzanne_Collins young-adult fiction dystopian dystopia fantasy ya science-fiction sci-fi series favourites romance adventure hunger-games book-club teen read-in-2012 post-apocalyptic my-books the-hunger-games favorite-books'

Some shelf tags are still present (`favourites`, `read-in-2012`, `my-books`):
a fixed list cannot anticipate every spelling. Notebook 03 replaces it with a
regular expression.

### 2.2 TF-IDF vectors and cosine similarity

`TfidfVectorizer` turns each document into a vector with one dimension per
token (author or tag). TF-IDF gives a high weight to tokens that are frequent
in the book but rare in the catalogue (`dystopia`) and a low weight to tokens
present everywhere (`fantasy`). The custom `token_pattern` splits on spaces only,
so that tags such as `young-adult` stay whole.

TF-IDF vectors are L2-normalised, so the dot product computed by `linear_kernel`
is exactly the **cosine similarity**. Returning the *n* most similar books is
equivalent to a k-nearest-neighbours search with the cosine metric.

In [4]:
tfidf = TfidfVectorizer(token_pattern=r"[^ ]+")
X = tfidf.fit_transform(books["content"])
X.shape

(10000, 12062)

In [5]:
def find_book(title):
    """Return the index of the most popular book whose title contains `title`."""
    matches = books[books["title"].str.contains(title, case=False, regex=False)]
    if matches.empty:
        raise ValueError(f"No book found for '{title}'")
    return matches.sort_values("ratings_count", ascending=False).index[0]


def recommend(title, n=10, exclude_same_author=False):
    """Recommend the n books most similar to `title` (cosine similarity on TF-IDF vectors)."""
    idx = find_book(title)
    scores = pd.Series(linear_kernel(X[idx], X).ravel(), index=books.index)
    scores = scores.drop(idx)

    if exclude_same_author:
        main_author = books.loc[idx, "authors"].split(",")[0].strip()
        same_author = books["authors"].str.contains(main_author, regex=False)
        scores = scores[~same_author.loc[scores.index]]

    best = scores.sort_values(ascending=False).head(n).index
    result = books.loc[best, ["title", "authors", "average_rating"]]
    return result.assign(similarity=scores.loc[best].round(3))

### 2.3 Results

In [6]:
recommend("The Hunger Games")

,title,authors,average_rating,similarity
506,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,4.49,0.875
19,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,4.03,0.858
16,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,4.30,0.792
716,The Hunger Games: Official Illustrated Movie C...,Kate Egan,4.51,0.626
6223,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,4.48,0.553
1354,The Hunger Games Tribute Guide,Emily Seife,4.40,0.495
9708,The Hunger Pains: A Parody,The Harvard Lampoon,3.89,0.474
712,"Prodigy (Legend, #2)",Marie Lu,4.29,0.442
2566,"Blood Red Road (Dust Lands, #1)",Moira Young,3.94,0.430
2915,"Partials (Partials Sequence, #1)",Dan Wells,3.94,0.427


In [7]:
recommend("Pride and Prejudice")

,title,authors,average_rating,similarity
170,Emma,"Jane Austen, Fiona Stafford",3.99,0.664
229,Persuasion,"Jane Austen, James Kinsley, Deidre Shauna Lynch",4.13,0.597
450,Northanger Abbey,"Jane Austen, Alfred MacAdam",3.80,0.529
6301,Lady Susan,"Jane Austen, Robert William Chapman",3.56,0.483
470,Mansfield Park,"Jane Austen, Kathryn Sutherland, Tony Tanner, ...",3.84,0.482
42,Jane Eyre,"Charlotte Brontë, Michael Mason",4.10,0.459
75,Sense and Sensibility,"Jane Austen, Tony Tanner, Ros Ballaster",4.06,0.450
4652,The Complete Novels,Jane Austen,4.55,0.446
102,The Count of Monte Cristo,"Alexandre Dumas, Robin Buss",4.21,0.406
65,Gone with the Wind,Margaret Mitchell,4.28,0.402


The recommendations are relevant, but **the author dominates**: author tokens
are rare in the catalogue, so TF-IDF gives them a high weight. For *Pride and
Prejudice*, most results are other Jane Austen novels; for *The Hunger Games*,
other volumes and companion books of the same franchise. A reader who just
finished a book probably already knows the rest of the series.

### 2.4 Excluding the author's other books

A simple fix is to drop books by the same main author (the first name in the
`authors` column; the others are often translators or illustrators):

In [8]:
recommend("The Hunger Games", exclude_same_author=True)

,title,authors,average_rating,similarity
716,The Hunger Games: Official Illustrated Movie C...,Kate Egan,4.51,0.626
6223,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,4.48,0.553
1354,The Hunger Games Tribute Guide,Emily Seife,4.40,0.495
9708,The Hunger Pains: A Parody,The Harvard Lampoon,3.89,0.474
712,"Prodigy (Legend, #2)",Marie Lu,4.29,0.442
2566,"Blood Red Road (Dust Lands, #1)",Moira Young,3.94,0.430
2915,"Partials (Partials Sequence, #1)",Dan Wells,3.94,0.427
11,"Divergent (Divergent, #1)",Veronica Roth,4.24,0.407
1412,"Under the Never Sky (Under the Never Sky, #1)",Veronica Rossi,4.01,0.404
3782,"Eve (Eve, #1)",Anna Carey,3.69,0.399


In [9]:
recommend("Pride and Prejudice", exclude_same_author=True)

,title,authors,average_rating,similarity
42,Jane Eyre,"Charlotte Brontë, Michael Mason",4.10,0.459
102,The Count of Monte Cristo,"Alexandre Dumas, Robin Buss",4.21,0.406
65,Gone with the Wind,Margaret Mitchell,4.28,0.402
62,Wuthering Heights,"Emily Brontë, Richard J. Dunn",3.82,0.396
3,To Kill a Mockingbird,Harper Lee,4.25,0.391
1621,A Connecticut Yankee in King Arthur's Court,Mark Twain,3.76,0.354
94,The Picture of Dorian Gray,"Oscar Wilde, Jeffrey Eugenides",4.06,0.354
558,Tess of the D'Urbervilles,"Thomas Hardy, Tim Dolin, Margaret R. Higonnet",3.77,0.352
1113,North and South,"Elizabeth Gaskell, Alan Shelston",4.13,0.347
426,The Importance of Being Earnest,Oscar Wilde,4.17,0.344


For *Pride and Prejudice*, the fix works well: the list now contains other
classic novels (*Jane Eyre*, *Wuthering Heights*, *North and South*…).

For *The Hunger Games*, it only partly works: the first four results are still
franchise companion books (movie guide, parody) written by other people, because
they share the franchise's tags (`hunger-games`, `the-hunger-games`). Genuine
young-adult dystopias (*Prodigy*, *Divergent*…) only come next. Detecting
franchises would require more than author names.

## Summary and limitations

| Model | Personalised? | Main weakness |
| --- | --- | --- |
| Popularity (weighted rating) | No | Same bestsellers for everyone |
| Content-based (TF-IDF + cosine) | Per book | Over-weights the author and the franchise |

Neither model uses the 6 million user ratings to learn tastes. The next notebook
groups the catalogue into genres with k-means; collaborative filtering is the
natural next step.